# Python + SQL: Working with Results

The previous lecture covered the basics of reading from a SQLite database in Python: connections, cursors, `execute`, `fetchall`, parameterized queries, and context managers. This notebook focuses on what to do with the results once you have them.

Along the way we will pick up a handful of new Python features: `sqlite3.Row`, `zip()`, list and dict comprehensions, `any()` / `all()`, and `collections.Counter`. These are standard tools that show up everywhere in real Python code. You now know enough Python to learn the rest on your own; think of this as a sampler of what is out there. For each one we will see it working first, then unpack the syntax.

## Setup

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

db_path = Path("data/5k-sales.db")
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://github.com/olearydj/INSY3010-Fall24/raw/main/notebooks/data/5k-sales.db"
    urlretrieve(url, db_path)
    print("Downloaded 5k-sales.db")
else:
    print(f"Using {db_path}")

This database has a single table, `sales`, holding 5000 rows of synthetic order data. We can confirm that by asking SQLite for its list of tables:

In [ ]:
import sqlite3

with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    print(cur.fetchall())

`sqlite_master` is a system table that SQLite maintains automatically inside every database. It holds one row for each table, index, view, and trigger, with columns for `type`, `name`, and the original `CREATE` statement that defined it. Because it is a regular table, you query it with ordinary SQL. Filtering on `type='table'` gives you just the user tables.

In the MySQL shell we used `SHOW TABLES` and `DESCRIBE tablename` to inspect structure. SQLite does not support those; instead it has a family of special statements called `PRAGMA` for metadata. `PRAGMA table_info(sales)` returns one row per column, with the column index, name, type, nullability, default, and primary-key flag:

In [ ]:
with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("PRAGMA table_info(sales)")
    for col in cur.fetchall():
        # col tuple: (cid, name, type, notnull, default, pk)
        print(f"  {col[1]:18s} {col[2]}")

Think of `PRAGMA` as SQLite's dialect of the same introspection job that `SHOW` / `DESCRIBE` did in MySQL. Every database system has its own commands for this; the SQL you write for querying data is portable, but the commands for inspecting schema often are not.

A quick peek at one row:

In [ ]:
with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("SELECT * FROM sales LIMIT 1")
    print(cur.fetchone())

### Data Dictionary

The columns we will use in this notebook:

| Column | Description |
|--------|-------------|
| `Region` | Geographic region (e.g. Asia, Europe) |
| `Country` | Country name |
| `Item_Type` | Product category |
| `Sales_Channel` | Online or Offline |
| `Order_ID` | Unique order identifier |
| `Units_Sold` | Quantity ordered |
| `Unit_Price` | Price per unit |
| `Unit_Cost` | Cost per unit |

## Wrapping a Query in a Function

When a query depends on a value, wrap it in a function. The function takes the value as a parameter and passes it safely into the query with a `?` placeholder.

In [ ]:
def orders_in_region(region):
    """Return all orders for a given region."""
    with sqlite3.connect(db_path) as con:
        cur = con.cursor()
        cur.execute("SELECT * FROM sales WHERE Region = ?", (region,))
        return cur.fetchall()

rows = orders_in_region("Asia")
print(f"Found {len(rows)} orders in Asia")
print(rows[0])

Two things worth noticing:

- The function opens the connection, runs the query, and returns the results. The `with` block handles closing the connection even if something goes wrong.
- The second argument to `execute()` is `(region,)`, a one-element tuple. The trailing comma is what makes it a tuple instead of just parentheses around a string.

This is the shape most of your project code will take: small functions that take parameters, run a query, and return rows.

## Getting Past `row[3]`

Results come back as tuples, so accessing a column means counting positions: `row[0]` is `Region`, `row[1]` is `Country`, and so on. That is error-prone and hard to read.

SQLite has a built-in fix: set `con.row_factory = sqlite3.Row` and rows come back as `Row` objects that support lookup by column name.

In [ ]:
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE Region = ? LIMIT 3", ("Asia",))
    rows = cur.fetchall()

for row in rows:
    print(f"{row['Order_ID']}: {row['Country']} - {row['Item_Type']}")

`sqlite3.Row` is a row factory class that comes with the `sqlite3` module. Setting it on the connection changes every row from a plain tuple into a `Row` object. `Row` still supports `row[0]` indexing and unpacking, so nothing you already know breaks; you just gain name access.

From here on, we will use `row_factory = sqlite3.Row` whenever the code benefits from named columns.

Under the hood, `Row` is saving us from writing the name-to-value pairing by hand. To see that, here is the manual equivalent using plain tuples. We pull column names from `cur.description` with a traditional loop, then pair them up with the row values:

In [ ]:
with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("SELECT Order_ID, Country, Units_Sold FROM sales LIMIT 1")
    row = cur.fetchone()

    # cur.description is a list of tuples; first element of each is the column name
    columns = []
    for col in cur.description:
        columns.append(col[0])

record = dict(zip(columns, row))
print(columns)
print(record)

`zip()` pairs up two sequences element by element: `zip(["a", "b"], [1, 2])` yields `("a", 1)` then `("b", 2)`. Passing those pairs to `dict()` turns them into key-value entries. You wrote a version of this by hand in an earlier homework problem. Normally you would just use `sqlite3.Row`; `dict(zip(columns, row))` is the pattern to recognize if you run into code that builds its records the manual way.

## Pulling One Column: List Comprehensions

Look at the column-extraction loop we just wrote:

```python
columns = []
for col in cur.description:
    columns.append(col[0])
```

We have been building lists like this all semester: start with an empty list, loop, append. Python has a shorthand for that pattern called a *list comprehension*. Let's use it to pull a single column out of a result set:

In [ ]:
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT DISTINCT Country FROM sales WHERE Region = ? ORDER BY Country", ("Asia",))
    rows = cur.fetchall()

countries = [row["Country"] for row in rows]
print(countries)

The expression `[row["Country"] for row in rows]` is the shorthand. The longhand would be:

```python
countries = []
for row in rows:
    countries.append(row["Country"])
```

Both produce the same list. The comprehension reads as "a list of `row['Country']` for each `row` in `rows`." You can also include a condition:

In [ ]:
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT Order_ID, Units_Sold FROM sales WHERE Region = ?", ("Asia",))
    rows = cur.fetchall()

big_orders = [row["Order_ID"] for row in rows if row["Units_Sold"] > 9000]
print(f"{len(big_orders)} orders over 9000 units")
print(big_orders[:5])

which is shorthand for:

```python
big_orders = []
for row in rows:
    if row["Units_Sold"] > 9000:
        big_orders.append(row["Order_ID"])
```

List comprehensions are one of the most common patterns in Python code. You will see them in almost every library's documentation.

### Why We Did It the Hard Way

A fair question at this point: if the shorthand is so much cleaner, why did we spend all semester writing the longhand?

Because the longhand *is* what is happening. The comprehension is the loop, just written compactly. Until you can write the empty-list-plus-`append` version in your sleep, the comprehension is just a black box that produces a list. Once you understand the pattern, the comprehension is a welcome shortcut; before that, it hides the mechanics you need to internalize.

The same logic applies to every shortcut in this notebook. They all compress a loop or a helper you have already written by hand.

## Lookup Tables: Dict Comprehensions

The same shorthand exists for dicts. This is useful when you want to look up records by a key instead of scanning a list every time.

Start with a simple case: mapping each `Order_ID` to its `Units_Sold`.

In [ ]:
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT Order_ID, Units_Sold FROM sales WHERE Region = ? LIMIT 5", ("Asia",))
    rows = cur.fetchall()

units_by_id = {row["Order_ID"]: row["Units_Sold"] for row in rows}
print(units_by_id)

The expression `{key: value for item in iterable}` is a *dict comprehension*. It is the dict version of the list comprehension we just saw. Longhand:

```python
units_by_id = {}
for row in rows:
    units_by_id[row["Order_ID"]] = row["Units_Sold"]
```

Now you can look up any order directly by its ID, without scanning the list. To demonstrate, we need *some* valid key to look up. We do not know any Order_ID off the top of our heads, so we just grab the first one the dict contains:

In [ ]:
some_id = next(iter(units_by_id))
print(f"Order {some_id} sold {units_by_id[some_id]} units")

That line deserves a quick breakdown:

- `iter(units_by_id)` turns the dict into an *iterator*, the same thing a `for` loop uses under the hood. For a dict, iterating yields the keys in insertion order.
- `next(...)` asks an iterator for its next value. Called once on a fresh iterator, it returns the first item.
- The net effect: "give me the first key of this dict, whatever it is."

We use it here only because we need *any* key to demo the lookup. In real code you would already have a specific Order_ID in hand.

### Nested: A Dict of Full Records

Sometimes you want more than one value per key. A `Row` converts to a dict with `dict(row)`, which gives you every column the query returned. Combine that with the dict comprehension and you get a full record-lookup structure.

In [ ]:
from pprint import pprint

with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE Region = ?", ("Asia",))
    rows = cur.fetchall()

# slice rows[:3] keeps the output small while we inspect the shape
orders_by_id = {row["Order_ID"]: dict(row) for row in rows[:3]}

pprint(orders_by_id)

Slices work inside a comprehension the same way they do in a longhand loop. `for row in rows[:3]` iterates over just the first three rows; the comprehension does not care where its iterable came from.

`orders_by_id` is a nested dict: the outer keys are order IDs, and each value is itself a dict of column-to-value pairs for that order. This is the standard in-memory shape for "I fetched a bunch of rows and want to work with them by key."

`pprint` ("pretty print") comes from the standard library. For nested structures like this one, it breaks the output across lines and indents each level so the shape is legible. Plain `print` would dump the whole dict on a single line.

Now that the shape is clear, pick any entry and drill in:

In [ ]:
some_id = next(iter(orders_by_id))
pprint(orders_by_id[some_id])
print()
print(f"Country: {orders_by_id[some_id]['Country']}")

One thing to notice: `Order_ID` appears twice for each order, once as the outer key and again as a field inside the inner dict. That is intentional and idiomatic. The outer dict is the index you use to find an order; each inner dict is a self-contained record you can hand off somewhere else (write to a file, pass to a function) without losing track of its own ID. The small duplication pays for that independence.

## Checking Rows: `any()` and `all()`

When you want to ask "does any row match this condition?" or "do all rows match?", Python has two built-ins that do exactly that.

In [ ]:
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE Region = ?", ("Asia",))
    rows = cur.fetchall()

any_huge = any(row["Units_Sold"] > 9000 for row in rows)
all_profitable = all(row["Unit_Price"] > row["Unit_Cost"] for row in rows)

print(f"Any order over 9000 units? {any_huge}")
print(f"Every unit sold at a markup? {all_profitable}")

`any(iterable)` returns `True` if at least one element is true, `False` otherwise. `all(iterable)` returns `True` only if every element is true. They short-circuit, meaning they stop as soon as the answer is known, which makes them efficient on large result sets.

They replace the flag-variable pattern you would otherwise write:

```python
any_huge = False
for row in rows:
    if row["Units_Sold"] > 9000:
        any_huge = True
        break
```

To see what `any()` and `all()` are actually checking, we can materialize the inner expression as a list first. Each row produces one `True` or `False`:

In [ ]:
# What any() / all() see, row by row
checks = [row["Units_Sold"] > 9000 for row in rows[:8]]
print(checks)
print("any:", any(checks))
print("all:", all(checks))

The expression `(row["Units_Sold"] > 9000 for row in rows)` inside `any(...)` is called a *generator expression*, a close cousin of the list comprehension. The only syntactic difference is parentheses instead of brackets. Unlike a list comprehension, it does not build a list up front; it produces values one at a time. That is ideal for `any` / `all`, which stop as soon as the answer is known and never need the full list in memory.

## Counting Values: `collections.Counter`

When you want "how many of each?" from a column, the standard library has a purpose-built tool.

In [ ]:
from collections import Counter

with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT Item_Type FROM sales WHERE Region = ?", ("Asia",))
    rows = cur.fetchall()

item_counts = Counter(row["Item_Type"] for row in rows)
print(item_counts)
print()
print("Top 3:", item_counts.most_common(3))

`Counter` is a specialized dict from the `collections` module. You pass it any iterable and it counts how many times each value appears. The `.most_common(n)` method returns the top `n` values as a sorted list of `(value, count)` pairs.

For comparison, here is the `count_values()` function you wrote earlier in the semester, applied to the same data, along with the loop needed to pick the top 3 values by count:

In [ ]:
def count_values(items):
    """Return a dict mapping each distinct value in items to its count."""
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0) + 1
    return counts

item_types = [row["Item_Type"] for row in rows]
counts = count_values(item_types)
print(counts)

# top 3 by count, the manual way
def by_count(pair):
    return pair[1]  # pair is (item, count); sort by count

top3 = sorted(counts.items(), key=by_count, reverse=True)[:3]
print(top3)

Same result as `Counter`, more lines, and we had to build the sort-by-count step ourselves. `Counter(...).most_common(3)` replaces all of it.

Worth being honest: the code above is not truly longhand. It already uses two shortcuts we have met: a list comprehension to build `item_types`, and `sorted(..., key=..., reverse=True)[:3]` to rank the counts. A fully manual version would expand the list comprehension back into an empty-list-plus-append loop, and would replace `sorted(..., key=...)` with something like a repeated scan for the current maximum, removing it from the dict, and appending it to a results list three times. That works, but it is significantly longer, harder to read, and easier to get wrong. Every shortcut we saw earlier in this notebook compresses exactly that kind of bookkeeping, which is part of why each one is worth recognizing.

One thing to notice in `count_values`: the line `counts[item] = counts.get(item, 0) + 1` is itself a shorthand. The longhand is the key-check pattern we used most of the semester:

```python
if item in counts:
    counts[item] += 1
else:
    counts[item] = 1
```

`dict.get(key, default)` returns the stored value if the key exists and the default if it does not. Combining it with `+ 1` collapses the whole if/else into a single line. That is on the Test 2 reference sheet and worth reaching for.

The argument to `Counter(...)` in the earlier cell is another generator expression. SQL's `GROUP BY ... COUNT(*)` does the same work on the database side. Use whichever fits the flow of your code.

## Exporting Results to CSV

Often the last step is saving a query result to disk so someone else can open it in Excel. Python's `csv` module handles this directly. You saw `csv.writer` briefly in the Files & Exceptions lecture; here it pairs naturally with `fetchall`.

### With Positional Rows

In [ ]:
import csv

with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("""
        SELECT Country, Item_Type, Units_Sold, Unit_Price
        FROM sales
        WHERE Region = ?
        ORDER BY Units_Sold DESC
        LIMIT 10
    """, ("Asia",))
    header = [col[0] for col in cur.description]
    rows = cur.fetchall()

with open("data/top_asia.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print("Wrote data/top_asia.csv")

`writerows(rows)` writes the entire result set in one call; no loop needed. It works directly on the list of tuples that `fetchall()` returns. The `newline=""` argument is a detail the `csv` docs recommend to avoid blank lines on Windows.

### With Dict Rows: `csv.DictWriter`

If your rows are dicts, or if you want the header handled for you, use `DictWriter`.

In [ ]:
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("""
        SELECT Country, Item_Type, Units_Sold, Unit_Price
        FROM sales
        WHERE Region = ?
        ORDER BY Units_Sold DESC
        LIMIT 10
    """, ("Asia",))
    rows = [dict(row) for row in cur.fetchall()]

with open("data/top_asia_dict.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("Wrote data/top_asia_dict.csv")

`DictWriter` takes a `fieldnames` argument that tells it which keys to write and in what order. `writeheader()` writes the column names as the first row; `writerows(rows)` writes all the data rows at once.

Rule of thumb: use `csv.writer` when you already have tuples; use `csv.DictWriter` when you have dicts and want self-documenting output.

## Putting It Together

Real code usually separates fetching from analysis. One function's job is to pull the data into a convenient shape; the rest of your code asks questions against it. Here is an upgraded version of the `orders_in_region` function from the start of the notebook, now returning a nested dict keyed by `Order_ID`:

In [ ]:
def orders_in_region(region):
    """Return a dict of {Order_ID: record} for all orders in a region."""
    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        cur = con.cursor()
        cur.execute("SELECT * FROM sales WHERE Region = ?", (region,))
        return {row["Order_ID"]: dict(row) for row in cur.fetchall()}

asia = orders_in_region("Asia")
print(f"{len(asia)} Asian orders")

With `asia` in hand, we can answer questions against it with one-liners. Each of these uses a different tool from this notebook:

In [ ]:
# unique countries represented
countries = sorted(set(o["Country"] for o in asia.values()))
print(f"{len(countries)} countries: {countries}")

In [ ]:
# top 3 item types by order count
top_items = Counter(o["Item_Type"] for o in asia.values()).most_common(3)
print("Top items:", top_items)

In [ ]:
# are there any very large orders?
has_bulk = any(o["Units_Sold"] > 9000 for o in asia.values())
print(f"Any order over 9000 units? {has_bulk}")

Notice what is happening structurally: the function went to the database exactly once. Every question after that runs against Python data already in memory. That is the pattern to reach for when you know you have several things to ask about the same result set.

## Problems

**★ 1. Unique Countries**

Write a function `unique_countries(region)` that returns a sorted list of the distinct countries appearing in a given region. Use a list comprehension somewhere in your solution.

In [ ]:
# your code here

In [ ]:
result = unique_countries("Asia")
assert isinstance(result, list)
assert result == sorted(result)
assert len(result) == len(set(result))  # all unique
assert len(result) > 0
print("All tests passed!")

#### Solution

In [ ]:
def unique_countries(region):
    """Return a sorted list of distinct countries in a region."""
    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        cur = con.cursor()
        cur.execute("SELECT DISTINCT Country FROM sales WHERE Region = ?", (region,))
        rows = cur.fetchall()
    return sorted([row["Country"] for row in rows])

print(unique_countries("Asia"))

**★★ 2. All Profitable?**

Write a function `all_profitable(region)` that returns `True` if every order in the given region has a `Unit_Price` greater than its `Unit_Cost`, and `False` otherwise. Use `all()` with a generator expression.

In [ ]:
# your code here

In [ ]:
result = all_profitable("Asia")
assert result is True or result is False

# Cross-check with a direct SQL count of unprofitable orders
with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute(
        "SELECT COUNT(*) FROM sales WHERE Region = ? AND Unit_Price <= Unit_Cost",
        ("Asia",),
    )
    bad_count = cur.fetchone()[0]

assert result == (bad_count == 0)
print("All tests passed!")

#### Solution

In [ ]:
def all_profitable(region):
    """Return True if every order in the region is priced above cost."""
    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        cur = con.cursor()
        cur.execute("SELECT Unit_Price, Unit_Cost FROM sales WHERE Region = ?", (region,))
        rows = cur.fetchall()
    return all(row["Unit_Price"] > row["Unit_Cost"] for row in rows)

print("Asia:", all_profitable("Asia"))
print("Europe:", all_profitable("Europe"))

**★★ 3. Items by Order ID**

Write a function `items_by_id(region)` that returns a dict mapping each `Order_ID` in the given region to its `Item_Type`. Use a dict comprehension.

In [ ]:
# your code here

In [ ]:
result = items_by_id("Asia")
assert isinstance(result, dict)

# Cross-check the size against a direct count
with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("SELECT COUNT(*) FROM sales WHERE Region = ?", ("Asia",))
    expected = cur.fetchone()[0]

assert len(result) == expected
# every value should be a string (the item type)
assert all(isinstance(v, str) for v in result.values())
print("All tests passed!")

#### Solution

In [ ]:
def items_by_id(region):
    """Return {Order_ID: Item_Type} for all orders in a region."""
    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        cur = con.cursor()
        cur.execute("SELECT Order_ID, Item_Type FROM sales WHERE Region = ?", (region,))
        return {row["Order_ID"]: row["Item_Type"] for row in cur.fetchall()}

sample = items_by_id("Asia")
print(f"{len(sample)} orders")
# show three entries
for oid in list(sample)[:3]:
    print(f"  {oid}: {sample[oid]}")

**★★★ 4. Truly Longhand `count_values`**

The `count_values` example in this notebook leaned on three shortcuts: a list comprehension to build `item_types`, `counts.get(item, 0) + 1` to accumulate counts, and `sorted(counts.items(), key=by_count, reverse=True)[:3]` to pick the top 3.

Rewrite the whole thing without those shortcuts. Produce the same `counts` dict and the same `top3` list, but use:

- a traditional loop with `.append()` to build `item_types`
- an explicit `if` / `else` with `counts[item] += 1` in your counting function (name it `count_values_longhand` to avoid a clash)
- a manual scan to find the top 3: repeatedly find the current maximum, record it, remove it, and repeat

Once you're done, compare your result to `Counter.most_common(3)` on the same data.

In [ ]:
# a fresh pull of rows to work with
with sqlite3.connect(db_path) as con:
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE Region = ?", ("Asia",))
    rows = cur.fetchall()

# your code here: build item_types, counts, and top3 with no shortcuts

In [ ]:
from collections import Counter

# compare against the shortcut version
expected_top3 = Counter(row["Item_Type"] for row in rows).most_common(3)

# your top3 should have the same (item, count) pairs as expected_top3,
# though the order among ties may differ. Compare as sets of pairs:
assert set(top3) == set(expected_top3)
assert sum(counts.values()) == len(rows)
print("All tests passed!")

#### Solution

In [ ]:
def count_values_longhand(items):
    """Count occurrences of each value in items without using .get()."""
    counts = {}
    for item in items:
        if item in counts:
            counts[item] += 1
        else:
            counts[item] = 1
    return counts

# build item_types with a traditional loop
item_types = []
for row in rows:
    item_types.append(row["Item_Type"])

counts = count_values_longhand(item_types)

# manual top-3: copy so we can remove without destroying the original
remaining = counts.copy()
top3 = []
for _ in range(3):
    # scan remaining for the current maximum
    best_key = None
    best_count = -1
    for key in remaining:
        if remaining[key] > best_count:
            best_key = key
            best_count = remaining[key]
    # record the winner and remove it so the next pass finds the runner-up
    top3.append((best_key, best_count))
    del remaining[best_key]

print(top3)

Each of the three shortcuts replaces a block like one of the ones above. That's what you're getting when you reach for a list comprehension, `dict.get`, or `sorted(..., key=..., reverse=True)[:n]`: not magic, just compression of patterns you already know how to write.

## Summary

Read-side patterns covered here:

| Pattern | Syntax |
|---------|--------|
| Parameterized query | `cur.execute(sql, (val,))` |
| Named column access | `con.row_factory = sqlite3.Row`; `row["Country"]` |
| Row to dict | `dict(row)` or `dict(zip(columns, row))` |
| Extract one column | `[row["Country"] for row in rows]` |
| Filter while extracting | `[row["Order_ID"] for row in rows if row["Units_Sold"] > 9000]` |
| Lookup table by key | `{row["Order_ID"]: row["Units_Sold"] for row in rows}` |
| Nested record dict | `{row["Order_ID"]: dict(row) for row in rows}` |
| Any / all row check | `any(cond for row in rows)`; `all(cond for row in rows)` |
| Frequency by value | `Counter(row["Item_Type"] for row in rows)` |
| Top n by count | `counter.most_common(n)` |
| Export rows to CSV | `csv.writer(f).writerows(rows)` |
| Export dicts to CSV | `csv.DictWriter(f, fieldnames=...)` |

New features to put in your toolkit:

- `sqlite3.Row` for named column access
- `zip()` for pairing two sequences into pairs
- List comprehensions for building lists in one line
- Dict comprehensions for building dicts (including lookup tables) in one line
- `any()` / `all()` for row-level boolean checks
- `collections.Counter` for frequency counts
- `csv.DictWriter` for CSV output with headers

None of these are required to solve a project task; loops and dicts work fine. They are here so you recognize them in other people's code and reach for them when they fit.

---

Auburn University / Industrial and Systems Engineering  
INSY 3010 / Programming and Databases for ISE / Spring 2026  
© Copyright 2026, Danny J. O'Leary.  
For licensing, attribution, and information: [GitHub INSY3010](https://github.com/olearydj/INSY3010)